# DFD Staffing Capacity & Operational Risk Analysis

## Project Questions

1. On a given day, were we understaffed, adequately staffed, or overstaffed?

2. Does this pattern repeat by shift (A/B/C)?

3. What does staffing look like on average (monthly / yearly)?

4. Are we compensating for understaffing with hirebacks?

5. Do these patterns justify hiring more people (or not)?

6. How much has hirebacks cost DFD in overtime pay per month and on a yearly basis on average?

## Issues

1. using telestaff to fill in the rows automatically so that Chief Eaton wouldn't have to keep updating the spreadsheet?
2. Where are the default values located? Telestaff?

#### Abbreviations

1. OPS - Operations
2. OOS - Out of Service
3. FLMA PTD - Family Partental Leave PT Disability
4. FTEs - Full Time Employees
5. VAC - Vacation
6. PPL - Paid Parental Leave
7. FMLA - Family Medical Leave Act
8. USAR - Urban Search and Rescue
9. TNG - Training (should actually be TRG)
10. MIL - Military leave
11. REASS - Reassigned
12. App - Appartus
13. LWOP - Leave Without Pay

#### Providing Clarity

1. Total Shift FTEs (Maximum available space)
2. Current Shift FTEs (Actual people who were hired for the shift and are still here)
3. Total @ Work (Who showed up today) 

#### Importing Libraries

In [16]:
import pandas as pd       # For working with tabular data (DataFrames), cleaning, filtering, and analysis
import numpy as np        # For numerical operations, arrays, and handling missing or complex data

#### Importing Input File

In [17]:
# The full file path
file_path = "C:/Users/AmarachiOny/Desktop/DFD_Projects/Python/DFD_Staffing_Analysis/input/Staffing_Tracker_2023_Copy_Sheet1.csv"

# Load the CSV file
df = pd.read_csv(file_path)

# Print the column names
print("Column names:")
print(df.columns.tolist())

Column names:
['Date', 'Shift', 'Total Shift FTEs', 'Current Shift FTEs', 'Total @ Work', 'Total Absences', 'VAC', 'SICK', 'Light Duty', 'PPL', 'FMLA PTD', 'COVID', 'USAR', 'TNG', 'MIL', 'Other', 'REASS', 'Hireback Total', 'FF Hireback', 'Driver Hireback', 'Captain Hireback', 'BC Hireback', 'Total Vehicles', 'App 4 Staffed', 'APP OOS', 'Support OOS', 'Total Ops FTEs', 'OPS VAC    ###', 'Notes', 'Unnamed: 29', 'Unnamed: 30', 'Unnamed: 31']


#### Dropping Columns

In [18]:
df = df.drop(columns=['Notes', 'Unnamed: 29', 'Unnamed: 30', 'Unnamed: 31'])

# Print the updated column names
print("Updated column names:")
print(df.columns.tolist())

Updated column names:
['Date', 'Shift', 'Total Shift FTEs', 'Current Shift FTEs', 'Total @ Work', 'Total Absences', 'VAC', 'SICK', 'Light Duty', 'PPL', 'FMLA PTD', 'COVID', 'USAR', 'TNG', 'MIL', 'Other', 'REASS', 'Hireback Total', 'FF Hireback', 'Driver Hireback', 'Captain Hireback', 'BC Hireback', 'Total Vehicles', 'App 4 Staffed', 'APP OOS', 'Support OOS', 'Total Ops FTEs', 'OPS VAC    ###']


#### Row Count and Preview

In [19]:
# Print the number of rows
print(f"Total number of rows: {len(df)}")

# Display the head
display(df.head())

Total number of rows: 1635


,Date,Shift,Total Shift FTEs,Current Shift FTEs,Total @ Work,Total Absences,VAC,SICK,Light Duty,PPL,...,FF Hireback,Driver Hireback,Captain Hireback,BC Hireback,Total Vehicles,App 4 Staffed,APP OOS,Support OOS,Total Ops FTEs,OPS VAC ###
0,7/1/2021,C,116,116,93,28,13,8,1,0,...,NaN,NaN,NaN,NaN,42,1,1,1,389,NaN
1,7/2/2021,A,115,115,93,24,14,5,2,0,...,NaN,NaN,NaN,NaN,42,0,0,0,389,NaN
2,7/3/2021,C,116,116,90,31,14,9,1,0,...,NaN,NaN,NaN,NaN,42,1,3,1,389,NaN
3,7/4/2021,A,115,115,92,25,16,5,1,0,...,NaN,NaN,NaN,NaN,42,0,1,1,389,NaN
4,7/5/2021,B,117,117,87,31,14,9,2,6,...,NaN,NaN,NaN,NaN,42,0,5,1,389,NaN


#### Standardize Date

In [20]:
# Date Formatting
df['Date'] = pd.to_datetime(df['Date'], errors='coerce')

# Full date (MM/DD/YYYY)
df['Full_Date'] = df['Date'].dt.strftime('%m/%d/%Y')

# Day number with suffix (1st, 2nd, 3rd, 4th...)
def day_suffix(day):
    if 11 <= day <= 13:
        return f"{day}th"
    last_digit = day % 10
    if last_digit == 1:
        return f"{day}st"
    elif last_digit == 2:
        return f"{day}nd"
    elif last_digit == 3:
        return f"{day}rd"
    else:
        return f"{day}th"

df['Day'] = df['Date'].dt.day
df['Day_With_Suffix'] = df['Day'].apply(day_suffix)

# Month name
df['Month'] = df['Date'].dt.month_name()

# Year
df['Year'] = df['Date'].dt.year

# Day of week
df['Day_Of_Week'] = df['Date'].dt.day_name()

#### Standardize Shift

In [21]:
# Shift Formatting
df['Shift'] = (
    df['Shift']
    .astype(str)
    .str.strip()
    .str.upper()
)

#### Force Numeric Columns

In [22]:
numeric_cols = [
    'Total Shift FTEs',
    'Current Shift FTEs',
    'Total @ Work',
    'Total Absences',
    'VAC',
    'SICK',
    'Light Duty',
    'PPL',
    'FMLA PTD',
    'COVID',
    'USAR',
    'TNG',
    'MIL',
    'Other',
    'REASS',
    'Hireback Total',
    'FF Hireback',
    'Driver Hireback',
    'Captain Hireback',
    'BC Hireback',
    'Total Vehicles',
    'App 4 Staffed',
    'APP OOS',
    'Support OOS',
    'Total Ops FTEs',
    'OPS VAC    ###'
]

for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)

#### Derived Metrics 1: Staffing Utilization Percentage

In [23]:
# Total Shift FTEs (available slots to occupy) --> Current Shift FTEs (actuall filled slots by employees) --> Total @ Work (who showed up today)

# Staffing Utilization is about whether planned staffing was met

# Thresholds:
#🟢 ≥ 90% → Healthy
#🟡 90–80% → Strained
#🔴 < 80% → Critical

# Staffing Utilization
df['Staffing_Utilization_Pct'] = (
    df['Total @ Work'] / df['Total Shift FTEs'] * 100
).replace([np.inf, -np.inf], np.nan)

# Staffing Status Flag
def staffing_flag(util):
    if util >= 90:
        return 'Healthy'
    elif util >= 80:
        return 'Strained'
    else:
        return 'Critical'

df['Staffing_Status'] = df['Staffing_Utilization_Pct'].apply(staffing_flag)

# Power BI stuff
# Bar Chart: Average Staffing by Shift? Which Shift is consistently struggling? Is a shift e.g B-Shift consistently red in summer months?

#### Derived Metrics 2: Coverage Gap

In [24]:
# Coverage Gap/ FTEs Slots to fill up
df['Coverage_Gap'] = df['Total Shift FTEs'] - df['Total @ Work'] 

#### Derived Metrics 3: Absence Burden Percentage

In [25]:
# Absence Burden Percentage = (Total Absences ÷ Total Current FTEs (Actual people who were hired for the shift and are still here)) * 100
df['Absence_Burden_Pct'] = (
    df['Total Absences'] / df['Current Shift FTEs'] * 100
).replace([np.inf, -np.inf], np.nan)

# Compare absences by shift, compare absences by month, and identify seasonal patterns

# Stacked Bar: Absence Reasons, Stack: VAC, SICK, FMLA, PPL, Light Duty, MIL
# This helps to argue: “This isn’t laziness — it’s structural leave.”

#### Derived Metrics 4: Hireback Dependency

In [26]:
# Hireback Dependency
df['Hireback_Pressure_Pct'] = (
    df['Hireback Total'] / df['Total Shift FTEs'] * 100
).replace([np.inf, -np.inf], np.nan)

# This answers What % of planned staffing required overtime?

# Less Hirebacks = Less overtime being paid. DFD would rather have more employees to fill those role than to be hiring hirebacks
# from left to right

#### Derived Metrics 5: Hireback Used Flag

In [27]:
# Hireback Used Flag
df['Hireback_Used'] = np.where(df['Hireback Total'] > 0, 'Yes', 'No')

#### Final Column Order (Power BI Friendly)

In [28]:
final_columns = [
    'Full_Date',
    'Day_With_Suffix',
    'Month',
    'Year',
    'Day_Of_Week',
    'Shift',
    'Total Shift FTEs',
    'Current Shift FTEs',
    'Total @ Work',
    'Total Absences',
    'Coverage_Gap',
    'Staffing_Utilization_Pct',
    'Staffing_Status',
    'Absence_Burden_Pct',
    'Hireback Total',
    'Hireback_Pressure_Pct',
    'Hireback_Used',
    'FF Hireback',
    'Driver Hireback',
    'Captain Hireback',
    'BC Hireback',
    'VAC',
    'SICK',
    'Light Duty',
    'PPL',
    'FMLA PTD',
    'MIL',
    'USAR',
    'TNG',
    'REASS',
    'APP OOS',
    'Support OOS',
    'Total Ops FTEs'
]

df_final = df[final_columns]

#### Preview Clean Data

In [29]:
display(df_final.sample(5))

print("\n📌 Updated Column Names:")
print(df.columns.tolist())

,Full_Date,Day_With_Suffix,Month,Year,Day_Of_Week,Shift,Total Shift FTEs,Current Shift FTEs,Total @ Work,Total Absences,...,Light Duty,PPL,FMLA PTD,MIL,USAR,TNG,REASS,APP OOS,Support OOS,Total Ops FTEs
460,10/04/2022,4th,October,2022,Tuesday,B,134,119,98,21,...,3,3,0,0,0,0,0.0,0,1,389
1586,11/03/2025,3rd,November,2025,Monday,C,133,115,98,21,...,3,1,0,0,0,1,1.0,0,0,354
92,10/01/2021,1st,October,2021,Friday,C,122,122,101,24,...,3,0,0,0,0,2,4.0,1,1,389
775,08/15/2023,15th,August,2023,Tuesday,B,119,110,87,26,...,4,2,1,0,0,0,0.0,0,1,350
724,06/25/2023,25th,June,2023,Sunday,B,123,121,97,30,...,2,2,1,0,0,1,1.0,2,2,361



📌 Updated Column Names:
['Date', 'Shift', 'Total Shift FTEs', 'Current Shift FTEs', 'Total @ Work', 'Total Absences', 'VAC', 'SICK', 'Light Duty', 'PPL', 'FMLA PTD', 'COVID', 'USAR', 'TNG', 'MIL', 'Other', 'REASS', 'Hireback Total', 'FF Hireback', 'Driver Hireback', 'Captain Hireback', 'BC Hireback', 'Total Vehicles', 'App 4 Staffed', 'APP OOS', 'Support OOS', 'Total Ops FTEs', 'OPS VAC    ###', 'Full_Date', 'Day', 'Day_With_Suffix', 'Month', 'Year', 'Day_Of_Week', 'Staffing_Utilization_Pct', 'Staffing_Status', 'Coverage_Gap', 'Absence_Burden_Pct', 'Hireback_Pressure_Pct', 'Hireback_Used']


#### Input and Output Fields Summary

##### Separating dataframe by year and Export for Power BI

In [30]:
import pandas as pd

# Base output folder
base_output_dir = r"C:/Users/AmarachiOny/Desktop/DFD_Projects/Python/DFD_Staffing_Analysis/output"

# Valid years
valid_years = [2021, 2022, 2023, 2024, 2025]

# Export one CSV per year
for year in valid_years:
    df_year = df_final[df_final["Year"] == year]

    if not df_year.empty:
        output_path = (
            base_output_dir
            + f"/DFD_Staffing_Fact_Table_{year}.csv"
        )

        df_year.to_csv(output_path, index=False)

        print(f"\n✅ Fact Table for {year} Exported for Power BI")
        print(f"Rows: {len(df_year)}")
        print(f"Saved to: {output_path}")

print("\n🎉 All yearly fact tables exported successfully.")


✅ Fact Table for 2021 Exported for Power BI
Rows: 184
Saved to: C:/Users/AmarachiOny/Desktop/DFD_Projects/Python/DFD_Staffing_Analysis/output/DFD_Staffing_Fact_Table_2021.csv

✅ Fact Table for 2022 Exported for Power BI
Rows: 365
Saved to: C:/Users/AmarachiOny/Desktop/DFD_Projects/Python/DFD_Staffing_Analysis/output/DFD_Staffing_Fact_Table_2022.csv

✅ Fact Table for 2023 Exported for Power BI
Rows: 365
Saved to: C:/Users/AmarachiOny/Desktop/DFD_Projects/Python/DFD_Staffing_Analysis/output/DFD_Staffing_Fact_Table_2023.csv

✅ Fact Table for 2024 Exported for Power BI
Rows: 366
Saved to: C:/Users/AmarachiOny/Desktop/DFD_Projects/Python/DFD_Staffing_Analysis/output/DFD_Staffing_Fact_Table_2024.csv

✅ Fact Table for 2025 Exported for Power BI
Rows: 355
Saved to: C:/Users/AmarachiOny/Desktop/DFD_Projects/Python/DFD_Staffing_Analysis/output/DFD_Staffing_Fact_Table_2025.csv

🎉 All yearly fact tables exported successfully.
